# Julia notebook to compute the symbolic solution to the frictional geostrophic equations with a specified buoyancuy field and surface wind stress.

twnh Nov '25

This notebook solves

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the surface wind stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H(x,y)) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This code derives the equation satisfied by the surface pressure field $p_s(x,y)$.

In [ ]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit
notebook_name = "PlanetaryGeostrophicTheory_v0.1.ipynb"

### Define symbols and parameter values

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x)
geometry_params = (H(x,y), z, ξ)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀, ϕ  = symbols("ν₀ ϕ",  real=true, positive=true) # Viscosity parameters
τs     = SymFunction("τs",    complex=true)         # Complex surface wind stress
psg    = SymFunction("psg",   complex=true)         # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)
pbarog = SymFunction("pbarog",complex=true)         # Baroclinic (NOT bottom) pressure gradient (∂/∂x + i ∂/∂y) (p(x,y,z) - pₛ(x,y))
pbotg  = SymFunction("pbotg", complex=true)         # Bottom (NOT baroclinic) pressure gradient (∂/∂x + i ∂/∂y)  p(x,y,z=-H) 
pg     = psg(x,y) + pbarog(x,y,z)                   # Total pressure gradient (∂/∂x + i ∂/∂y) p(x,y,z)
b      = SymFunction("b",     real=true)            # Buoyancy field b(x,y,z)
χt     = SymFunction("χt", complex=true)            # \chi term = (∂/∂y + i ∂/∂x) χ(x,y)

# Define symbolic viscosity here:
ν = ν₀                                              # Constant viscosity profile
uv_params = (f, ϵ, ν)

### Function to solve the frictional geostrophic equation using a Green's function:

In [ ]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, ϵ, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                            # Unknown coefficient in the Green's function solution

    #0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ϵ^2 *  ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))


    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final simplify (to cancel constants). Avoid simplify in general because it's not always reproducible.
    G = simplify(G)
    return G
end ;

### Compute the G's function:

In [ ]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
Guv = Guv_sym.subs(f,ϕ^2 * ϵ^2 * ν₀)
display("Simplified Guv(z,ξ)")
display(Guv)

Guv0 = Guv.subs(ξ,0)
display("Guv(z,0)")
display(Guv0)

# Used below to check consistency:
tmp = integrate(expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)

### Symbolic computation of flow $(u(z),v(z)), U, V, \tau_b$

In [ ]:
# Compute flow field u(z), v(z):
𝔲1 = integrate(expand(Guv * pg.subs(z,ξ)),(ξ,-H(x,y),0))
max_obj = sympy.Max(z, -H(x, y))
𝔲1 = 𝔲1.subs(max_obj,z)
display("Pressure-driven flow field 𝔲₁(x,y,z):")
display(simplify(𝔲1))

#Check consistency with simplified integral where pbarog = 0 (which is used in Example_barotropic_GeostrophicFlow_v0.3.ipynb):
tmp = Guv_int_wrt_ξ * psg(x,y)  
@assert simplify(tmp - 𝔲1.subs(pbarog(x,y,ξ),0)) == 0       

𝔲2 = - Guv0 * τs(x,y)
display("Stress-driven   flow field 𝔲₂(x,y,z):")
display(simplify(𝔲2))
𝔲 = 𝔲1 + 𝔲2

# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * f * 𝔲1 + ϵ^2 * diff(diff(ν * 𝔲1,z),z)
tmp1 = simplify(tmp1.subs(ϕ,sqrt(f/ν₀)/ϵ))
tmp2 = -im * f * 𝔲2 + ϵ^2 * diff(diff(ν * 𝔲2,z),z)
tmp2 = simplify(tmp2.subs(ϕ,sqrt(f/ν₀)/ϵ))
@assert tmp1 + tmp2 == pg

# Compute depth-integrated flow:
𝔘 = integrate(expand(𝔲),(z,-H(x,y),0))
display("Depth-integrated flow field 𝔘(x,y):")
display(simplify(𝔘))

# Compute bottom stress on fluid:
τb = - ϵ^2 * ν * diff(𝔲,z).subs(z,-H(x,y))
display("Bottom stress on fluid -ν ϵ^2 d/dz u(x,y,z) @ z = -H:")
display(simplify(τb))

# Check expression for surface stress on fluid:
@assert simplify(diff(𝔲1,z).subs(z,0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ϵ^2 * ν * diff(𝔲,z).subs(z,0)
@assert simplify(tmp - τs(x,y)) == 0

# Compute baroclinic potential energy $\chi(x,y):
χ = integrate(- z * b(x,y,z),(z,-H(x,y),0))

### Form final equation for $\frak{U}$:

In [ ]:
lhs = im * f * 𝔘
rhs = - H(x,y)*pbotg(x,y) - χt(x,y) + τs(x,y) - τb 
println()
display("Final equation linking surface pressure pₛ(x,y) to windstress:")
display(simplify(lhs))
display("=")
display(simplify(rhs))

# Solve for surface pressure gradient
eqn = simplify(expand(lhs - rhs).subs(f,ϕ^2 * ϵ^2 * ν₀))
eqn = eqn.subs(pbotg(x,y),pg.subs(z,-H(x,y)))
psg_soln = simplify(solve(eqn,psg(x,y))[1])
println()
display("Surface pressure gradient expression:")
display(psg_soln)

In [ ]:
# Check intermediate results from LaTex derivation
tmp = sympy.sqrt(im)*ϕ
@assert simplify(1+(((exp(tmp*H(x,y)) - 1)^2) / (1 + exp(2*tmp*H(x,y)))) + (2*exp(tmp*H(x,y)))/ (1 + exp(2*tmp*H(x,y)))) == 2
A = exp(-tmp*ξ)*(exp(tmp*ξ) - exp(tmp*H(x,y)))*(exp(tmp*(ξ + H(x,y))) - 1) + exp(-tmp*(ξ - H(x,y)))*(1 + exp(2*tmp*ξ))
B = exp(tmp*(ξ + H(x,y))) - exp(2*tmp*H(x,y)) - 1 + 2*exp(tmp*(H(x,y) - ξ)) + exp(tmp*(ξ + H(x,y)))
@assert simplify(A-B) == 0